# Build merged custom-data topic assignments

This notebook rebuilds `custom_data_all_models_topic_assignments.json` and `.csv` from per-model `document_assignments.csv` files, using exact `document_text` matching to recover the correct `doc_id` and `content`.

In [ ]:
import csv
import json
import re
from collections import defaultdict, deque
from pathlib import Path

# Paths (works whether notebook is run from repo root or notebooks/)
CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((c for c in CANDIDATES if (c / 'custom_data').exists()), None)
if REPO_ROOT is None:
    raise RuntimeError('Could not find repo root containing custom_data/')

CUSTOM_DATA_DIR = REPO_ROOT / 'custom_data'
OUT_DIR = REPO_ROOT / 'data_out' / 'custom_data'
OUT_JSON = OUT_DIR / 'custom_data_all_models_topic_assignments.json'
OUT_CSV = OUT_DIR / 'custom_data_all_models_topic_assignments.csv'
DATASET_NAME = 'custom_data'
RUN_INDEX = '0'

# 1) Load source docs in the same order as GenericDataset (sorted filenames, skip empty content)
docs = []
for p in sorted(CUSTOM_DATA_DIR.glob('*.json')):
    try:
        obj = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    content = (obj.get('content') or '').strip()
    if not content:
        continue
    docs.append({
        'doc_index': len(docs),
        'doc_id': str(obj.get('_id', p.stem)),
        'content': content,
    })

if not docs:
    raise RuntimeError(f'No non-empty docs found in {CUSTOM_DATA_DIR}')

# 2) Find all per-model assignment files
run_folders = sorted({p.parent for p in OUT_DIR.rglob('document_assignments.csv')})
if not run_folders:
    raise RuntimeError(f'No document_assignments.csv files found under {OUT_DIR}')

def model_name_from_folder(folder_name: str, dataset_name: str):
    m = re.search(rf'_{re.escape(dataset_name)}_(.+)_equal$', folder_name)
    return m.group(1) if m else folder_name

# 3) Build doc_id -> topic mapping for each model
model_to_doc_topic = {}
for folder in run_folders:
    model_name = model_name_from_folder(folder.name, DATASET_NAME)
    assignment_file = folder / 'document_assignments.csv'

    with assignment_file.open(encoding='utf-8', newline='') as f:
        rows = [r for r in csv.DictReader(f) if r.get('run') == RUN_INDEX]

    # New queue per model so duplicate document texts are handled deterministically
    text_to_ids = defaultdict(deque)
    for d in docs:
        text_to_ids[d['content']].append(d['doc_id'])

    by_doc_id = {}
    missing = 0
    for r in rows:
        text = r.get('document_text', '')
        topic = r.get('llm_assigned_topic', '')
        q = text_to_ids.get(text)
        if not q:
            missing += 1
            continue
        by_doc_id[q.popleft()] = topic

    if missing:
        print(f'Warning: {model_name}: {missing} rows could not be mapped by text')
    if len(by_doc_id) != len(docs):
        print(f'Warning: {model_name}: mapped {len(by_doc_id)}/{len(docs)} docs')

    model_to_doc_topic[model_name] = by_doc_id

# 4) Assemble merged rows in source-doc order
model_names = sorted(model_to_doc_topic.keys())
rows_json = []
rows_csv = []
for d in docs:
    topics = {m: model_to_doc_topic[m].get(d['doc_id'], '') for m in model_names}
    rows_json.append({
        'doc_index': d['doc_index'],
        'doc_id': d['doc_id'],
        'content': d['content'],
        'topics_by_model': topics,
    })

    row = {'doc_index': d['doc_index'], 'doc_id': d['doc_id'], 'content': d['content']}
    row.update(topics)
    rows_csv.append(row)

# 5) Write outputs
OUT_DIR.mkdir(parents=True, exist_ok=True)
with OUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(rows_json, f, ensure_ascii=False, indent=2)

with OUT_CSV.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['doc_index', 'doc_id', 'content'] + model_names)
    writer.writeheader()
    writer.writerows(rows_csv)

print(f'Wrote: {OUT_JSON}')
print(f'Wrote: {OUT_CSV}')
print(f'Docs: {len(docs)}, Models: {len(model_names)}')

# Optional spot-check
target_id = '663db9c5277dee4cd2857f37'
match = next((r for r in rows_json if r['doc_id'] == target_id), None)
if match:
    print('Spot-check gpt-5.2:', match['topics_by_model'].get('gpt-5.2', '<missing>'))
else:
    print('Target doc not found:', target_id)
